In [2]:
import pandas as pd
from pathlib import Path
import numpy as np


In [3]:
BASE = Path().resolve().parent
RAW = BASE  / 'data' / 'original_data'
BLD = BASE/ 'data' / 'processed'
OUTPUT = BASE / 'data' / 'outputs'

In [19]:
affected = pd.read_parquet(BLD/'affected_companies.parquet')
score = pd.read_parquet(BLD / 'score2.parquet')
ev = pd.read_parquet(BLD /'ev.parquet')
df_fin = pd.read_parquet(BLD / 'financial_data_full_yearly.parquet')
df_fin.rename(columns={'Stkcd': 'stock_id'}, inplace=True)

In [20]:
df_fin

,stock_id,year,revenue_growth,size,turnover_rate,manage_exp_ratio,debt_ratio,nwc,roe,roa,capex_ratio
0,000002,2012,NaN,26.660278,0.272217,0.026963,0.121352,0.271752,0.196648,0.033134,0.000717
1,000002,2013,0.313263,26.895395,0.282590,0.022174,0.102637,0.236067,0.196610,0.031549,0.005090
2,000002,2014,0.081002,26.954552,0.287934,0.026659,0.095459,0.234362,0.178592,0.030970,0.013771
3,000002,2015,0.335828,27.138846,0.319893,0.024266,0.089555,0.207694,0.180862,0.029641,0.012624
4,000002,2016,0.229754,27.445504,0.289496,0.028279,0.122901,0.170099,0.185311,0.025308,0.004257
...,...,...,...,...,...,...,...,...,...,...,...
22136,603999,2015,NaN,21.374546,0.430250,0.114168,0.040667,0.719140,0.063853,0.052951,0.031337
22137,603999,2016,-0.090084,21.384553,0.387593,0.136939,0.005162,0.612731,0.051261,0.043490,0.008706
22138,603999,2017,0.051827,21.408195,0.398155,0.126282,0.005041,0.515102,0.044543,0.037951,0.012976
22139,603999,2018,-0.036826,21.417421,0.379971,0.119584,0.003996,0.486132,0.025284,0.021215,0.009615


In [5]:
def construct_did_data(score_df, affected_df, impact_col_name):
    # 选出当前门槛下的 treated 样本（只选 impact_category == 1）
    treated_stocks = affected_df[affected_df[impact_col_name] == 1]['Stkcd'].unique()
    excluded_stocks = affected_df[affected_df[impact_col_name].isin([2, 3])]['Stkcd'].unique()
    
    # 剔除非 treated 样本
    score_filtered = score_df[~score_df['stock_id'].isin(excluded_stocks)].copy()
    
    # 添加 treated 和 post
    score_filtered['treated'] = score_filtered['stock_id'].isin(treated_stocks).astype(int)
    score_filtered['post'] = (score_filtered['year'] >= 2018).astype(int)
    
    # 保留变量
    df_did = score_filtered[[
        'stock_id',
        'year',
        'IndustryCode',
        'treated',
        'post',
        'total_score_norm'
    ]]
    
    return df_did

In [6]:
def score_state_owned():
    df = pd.read_csv(RAW / 'csmar/董事会治理/董事会基本情况/BDT_ManaGovAbil.csv')
    df['stock_id'] = df['Symbol'].astype(str).str.zfill(6)
    df['year'] = pd.to_datetime(df['Enddate']).dt.year

    def parse_nature(value):
        if pd.isna(value):
            return 0
        elif value == 1:
            return 1  # 国企
        elif value == 0:
            return 0  # 非国企
        else:
            return 0

    df['soe'] = df['ContrshrNature'].apply(parse_nature)

    return df[['stock_id', 'year', 'soe']]


df_soe = score_state_owned()
df_soe

,stock_id,year,soe
0,000002,2012,1
1,000002,2013,1
2,000002,2014,1
3,000002,2015,1
4,000002,2016,1
...,...,...,...
21829,688368,2019,0
21830,688369,2019,0
21831,688388,2019,0
21832,688389,2019,0


In [7]:
def merge_all_sources(did_df, df_fin, ev):
    df = did_df.merge(df_fin, on=['stock_id', 'year'], how='left')
    df = df.merge(ev[['stock_id', 'year', 'ev', 'tobinq']], on=['stock_id', 'year'], how='left')
    df = df.merge(df_soe[['stock_id', 'year', 'soe']], on=['stock_id', 'year'], how='left')
    df.rename(columns={'total_score_norm': 'gov_score'}, inplace=True)
    df['firm_id'] = df['stock_id'].astype('category').cat.codes + 1
    df.drop(columns=['IndustryCode'], inplace=True, errors='ignore')
    return df

In [23]:
thresholds = {
    '10pct': 'impact_10pct',
    '9pct': 'impact_9pct',
    '8pct': 'impact_8pct',
    '7pct': 'impact_7pct',
    '6pct': 'impact_6pct',
    '5pct': 'impact_5pct',
    '4pct': 'impact_4pct',
    '3pct': 'impact_3pct',
    '2pct': 'impact_2pct',
    '1pct': 'impact_1pct'
}



for label, impact_col in thresholds.items():
    print(f"Processing threshold: {label}")
    
    # 构建 DID 样本
    did_df = construct_did_data(score, affected, impact_col)
    
    # 合并财务和估值数据
    merged_df = merge_all_sources(did_df, df_fin, ev)
    merged_df['revenue_growth'] = merged_df['revenue_growth'].replace([np.inf, -np.inf], np.nan)

    # 保存结果
    merged_df.to_parquet(BLD / f'did_data_{label}.parquet', index=False)
    merged_df.to_stata(BLD / f'did_data_{label}.dta', write_index=False)

Processing threshold: 10pct
Processing threshold: 9pct
Processing threshold: 8pct
Processing threshold: 7pct
Processing threshold: 6pct
Processing threshold: 5pct
Processing threshold: 4pct
Processing threshold: 3pct
Processing threshold: 2pct
Processing threshold: 1pct


In [17]:
bb.isna().sum()

stock_id           0
year               0
treated            0
post               0
gov_score          0
revenue_growth    56
size               0
debt_ratio         0
nwc                0
roe                0
roa                0
capex              0
ev                 0
tobinq             0
soe                0
firm_id            0
dtype: int64

In [202]:
year_counts = bb.groupby('stock_id')['year'].nunique()

# 筛出不是7年的公司（说明缺某年）
incomplete_firms = year_counts[year_counts < 7]

print("这些公司年份数据不完整：")
print(incomplete_firms)

这些公司年份数据不完整：
Series([], Name: year, dtype: int64)


In [18]:
did_df.groupby('stock_id')['year'].nunique().value_counts()


year
6    1327
Name: count, dtype: int64

In [164]:
df_fin

,stock_id,year,revenue_growth,size,debt_ratio,nwc,roe,roa
0,000001,2013,NaN,28.268519,0.004283,0.000000,0.135893,0.008051
1,000001,2014,0.377101,28.413304,0.019095,0.000000,0.151219,0.009057
2,000001,2015,0.312414,28.550167,0.084942,0.000000,0.135387,0.008721
3,000001,2016,0.126325,28.713990,0.089206,0.000000,0.111782,0.007652
4,000001,2017,0.004338,28.809206,0.105432,0.000000,0.104430,0.007138
...,...,...,...,...,...,...,...,...
26419,605388,2020,NaN,21.498782,0.000000,0.774547,0.110747,0.098464
26420,605389,2020,NaN,20.749945,0.000000,0.440589,0.329943,0.229648
26421,605398,2020,NaN,20.312167,0.077022,0.617536,0.261131,0.163653
26422,605399,2020,NaN,20.943559,0.000000,0.683549,0.112488,0.102338


In [113]:
df_fin.rename(columns={'Stkcd': 'stock_id'}, inplace=True)

# 左连接，以主表 df1_new 为准
df_merged = df_did.merge(df_fin, on=['stock_id', 'year'], how='left')

In [114]:
df_merged

,stock_id,year,IndustryCode,treated,post,total_score_norm,revenue_growth,size,debt_ratio,nwc,roe,roa
0,000012,2014,C30,0,0,0.678571,-0.089127,23.450098,0.153159,-0.275356,0.095987,0.051551
1,000012,2015,C30,0,0,0.750000,0.054849,23.474195,0.345932,-0.185206,0.069697,0.034020
2,000012,2016,C30,0,0,0.392857,0.207673,23.565078,0.318224,-0.251350,0.102155,0.046523
3,000012,2017,C30,0,0,0.571429,0.212313,23.695474,0.269196,-0.129012,0.097580,0.042252
4,000012,2018,C30,0,1,0.571429,-0.024766,23.673699,0.274056,-0.066293,0.049759,0.023698
...,...,...,...,...,...,...,...,...,...,...,...,...
8545,603998,2015,C27,0,0,0.642857,0.127391,20.787812,0.000000,0.565850,0.102409,0.085008
8546,603998,2016,C27,0,0,0.607143,0.140214,20.963072,0.051298,0.248604,0.075132,0.054882
8547,603998,2017,C27,0,0,0.500000,0.349106,21.040223,0.048283,0.168147,0.060611,0.043469
8548,603998,2018,C27,0,1,0.714286,0.458272,21.219582,0.109578,0.139770,0.069255,0.044436


In [115]:
df_merged.isna().sum()

stock_id             0
year                 0
IndustryCode         0
treated              0
post                 0
total_score_norm     0
revenue_growth      84
size                 0
debt_ratio           0
nwc                  0
roe                  0
roa                  0
dtype: int64

In [79]:
df_merged2 = pd.merge(
    df_merged,
    ev[['stock_id', 'year', 'ev']],
    on=['stock_id', 'year'],
    how='left'
)
df_merged2

,stock_id,year,IndustryCode,treated,post,total_score_norm,revenue_growth,size,debt_ratio,nwc,roe,roa,ev
0,000012,2014,C30,0,0,0.678571,-0.089127,23.450098,0.153159,-0.275356,0.095987,0.051551,1.941370e+10
1,000012,2015,C30,0,0,0.750000,0.054849,23.474195,0.345932,-0.185206,0.069697,0.034020,2.718260e+10
2,000012,2016,C30,0,0,0.392857,0.207673,23.565078,0.318224,-0.251350,0.102155,0.046523,2.494733e+10
3,000012,2017,C30,0,0,0.571429,0.212313,23.695474,0.269196,-0.129012,0.097580,0.042252,2.081444e+10
4,000012,2018,C30,0,1,0.571429,-0.024766,23.673699,0.274056,-0.066293,0.049759,0.023698,1.341474e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8545,603998,2015,C27,0,0,0.642857,0.127391,20.787812,0.000000,0.565850,0.102409,0.085008,6.700002e+09
8546,603998,2016,C27,0,0,0.607143,0.140214,20.963072,0.051298,0.248604,0.075132,0.054882,7.331350e+09
8547,603998,2017,C27,0,0,0.500000,0.349106,21.040223,0.048283,0.168147,0.060611,0.043469,4.747806e+09
8548,603998,2018,C27,0,1,0.714286,0.458272,21.219582,0.109578,0.139770,0.069255,0.044436,1.863185e+09


In [128]:
df_merged3 = pd.merge(
    df_merged,
    ev3[['stock_id', 'year', 'ev', 'tobinq','tobinq2','tobinq3']],
    on=['stock_id', 'year'],
    how='left'
)
df_merged3.rename(columns={'total_score_norm': 'gov_score'}, inplace=True)
df_merged3['firm_id'] = df_merged3['stock_id'].astype('category').cat.codes + 1
df_merged3 = df_merged3.drop(columns=["IndustryCode", "roe", "roa"])

df_merged3



,stock_id,year,treated,post,gov_score,revenue_growth,size,debt_ratio,nwc,ev,tobinq,tobinq2,tobinq3,firm_id
0,000012,2014,0,0,0.678571,-0.089127,23.450098,0.153159,-0.275356,1.941370e+10,1.280512,1.431696,1.526540,1
1,000012,2015,0,0,0.750000,0.054849,23.474195,0.345932,-0.185206,2.718260e+10,1.773080,1.923578,2.033901,1
2,000012,2016,0,0,0.392857,0.207673,23.565078,0.318224,-0.251350,2.494733e+10,1.489147,1.636809,1.785718,1
3,000012,2017,0,0,0.571429,0.212313,23.695474,0.269196,-0.129012,2.081444e+10,1.191556,1.426640,1.540565,1
4,000012,2018,0,1,0.571429,-0.024766,23.673699,0.274056,-0.066293,1.341474e+10,0.818301,1.007007,1.087357,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8545,603998,2015,0,0,0.642857,0.127391,20.787812,0.000000,0.565850,6.700002e+09,6.780504,6.907012,7.711288,1425
8546,603998,2016,0,0,0.607143,0.140214,20.963072,0.051298,0.248604,7.331350e+09,6.054101,6.231999,7.598691,1425
8547,603998,2017,0,0,0.500000,0.349106,21.040223,0.048283,0.168147,4.747806e+09,3.560575,3.756575,4.599137,1425
8548,603998,2018,0,1,0.714286,0.458272,21.219582,0.109578,0.139770,1.863185e+09,1.327962,1.540918,1.842288,1425


In [126]:
df_merged3.isna().sum()

year                0
treated             0
post                0
gov_score           0
revenue_growth     84
size                0
debt_ratio          0
nwc                 0
ev                 26
tobinq             26
tobinq2           301
tobinq3           301
firm_id             0
dtype: int64

In [129]:
df_merged3[df_merged3['tobinq2'].isna()]

,stock_id,year,treated,post,gov_score,revenue_growth,size,debt_ratio,nwc,ev,tobinq,tobinq2,tobinq3,firm_id
18,000020,2014,0,0,0.321429,0.108514,20.874046,0.622163,0.511194,2.455115e+09,2.136275,NaN,NaN,4
44,000048,2016,0,0,0.500000,-0.321799,21.396076,0.088213,0.031286,1.470492e+10,7.649405,NaN,NaN,8
56,000050,2016,0,0,0.642857,0.019635,23.799625,0.059504,0.183750,2.346408e+10,1.288198,NaN,NaN,10
85,000066,2015,0,0,0.464286,-0.055301,24.398454,0.116530,0.095048,2.999358e+10,0.853634,NaN,NaN,15
103,000153,2015,0,0,0.500000,-0.085142,21.558468,0.227131,-0.014610,3.646273e+09,1.653228,NaN,NaN,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8502,603799,2014,0,0,0.428571,NaN,22.788141,0.420481,-0.085683,NaN,NaN,NaN,NaN,1418
8515,603889,2015,0,0,0.607143,0.057756,21.080971,0.003551,0.375988,4.880143e+09,3.584475,NaN,NaN,1420
8520,603899,2014,0,0,0.642857,NaN,21.283819,0.000575,0.230255,NaN,NaN,NaN,NaN,1421
8526,603969,2014,0,0,0.571429,NaN,20.951025,0.242876,0.386521,NaN,NaN,NaN,NaN,1422


In [131]:
df_merged3[df_merged3['tobinq2'].isna()].groupby(['year', 'treated', 'post']).size()


year  treated  post
2014  0        0       79
      1        0        2
2015  0        0       82
      1        0        4
2016  0        0       60
      1        0        4
2017  0        0       55
      1        0        4
2018  0        1        3
2019  0        1        7
      1        1        1
dtype: int64

In [133]:
df_merged3['tobinq_diff'] = df_merged3['tobinq2'] - df_merged3['tobinq']
df_merged3['tobinq_diff_pct'] = df_merged3['tobinq_diff'] / df_merged3['tobinq']
df_merged3[['tobinq_diff', 'tobinq_diff_pct']].dropna().describe()

,tobinq_diff,tobinq_diff_pct
count,8249.000000,8249.000000
mean,0.261158,0.182367
std,0.739557,0.205878
min,0.009062,0.001387
25%,0.152164,0.058780
50%,0.223458,0.117886
75%,0.320621,0.229147
max,63.883337,3.532895


In [134]:
large_diff = df_merged3[(df_merged3['tobinq_diff'].abs() > 1) | (df_merged3['tobinq_diff_pct'].abs() > 1)]
print(large_diff[['stock_id', 'year', 'tobinq', 'tobinq2', 'tobinq_diff', 'tobinq_diff_pct']])


     stock_id  year     tobinq    tobinq2  tobinq_diff  tobinq_diff_pct
84     000066  2014   0.295152   0.978179     0.683027         2.314152
86     000066  2016   0.438572   1.170343     0.731771         1.668530
124    000338  2018   0.473136   0.996685     0.523549         1.106549
149    000404  2019   0.417113   0.852428     0.435315         1.043638
191    000509  2019  12.629633  14.248070     1.618437         0.128146
...       ...   ...        ...        ...          ...              ...
7762   600875  2018   0.274970   0.930933     0.655963         2.385578
7763   600875  2019   0.325819   0.962129     0.636310         1.952953
7889   600983  2019   0.499620   1.003154     0.503534         1.007832
8182   601727  2018   0.491086   0.995908     0.504822         1.027972
8183   601727  2019   0.430314   0.942501     0.512187         1.190265

[88 rows x 6 columns]


In [105]:
stata_path = OUTPUT / "data_tobinq2.dta"
df_merged3.to_stata(stata_path, write_index=False)

In [96]:
df_merged2.isna().sum()

stock_id             0
year                 0
IndustryCode         0
treated              0
post                 0
total_score_norm     0
revenue_growth      84
size                 0
debt_ratio           0
nwc                  0
roe                  0
roa                  0
ev                  26
firm_id              0
dtype: int64

In [81]:
df_merged2[df_merged2['ev'].isna()]

,stock_id,year,IndustryCode,treated,post,total_score_norm,revenue_growth,size,debt_ratio,nwc,roe,roa,ev
4182,002734,2014,C26,0,0,0.642857,NaN,20.651354,0.343859,-0.036751,0.095632,0.051973,NaN
4200,002743,2014,C33,0,0,0.464286,NaN,21.837969,0.365997,-0.023806,0.072968,0.012762,NaN
4206,002745,2014,C39,0,0,0.500000,NaN,22.370519,0.112570,-0.146464,0.295321,0.083533,NaN
4212,002747,2014,C40,0,0,0.392857,NaN,19.933313,0.173613,0.237873,0.152709,0.096869,NaN
5574,300374,2014,C30,0,0,0.464286,NaN,20.667645,0.227297,0.181487,0.139260,0.065889,NaN
5730,300415,2014,C35,0,0,0.571429,NaN,20.966649,0.307822,0.012517,0.199601,0.059618,NaN
5736,300416,2014,C40,0,0,0.464286,NaN,19.894393,0.127136,0.304422,0.149759,0.091151,NaN
5742,300417,2014,C40,0,0,0.571429,NaN,19.216914,0.000000,0.675739,0.158787,0.135597,NaN
5748,300420,2014,C34,0,0,0.571429,NaN,19.612692,0.024289,0.436359,0.123645,0.094051,NaN
5754,300421,2014,C34,0,0,0.464286,NaN,20.282846,0.310672,0.015179,0.162854,0.081367,NaN


In [83]:
df_merged2['firm_id'] = df_merged2['stock_id'].astype('category').cat.codes + 1



In [84]:
stata_path = OUTPUT / "data.dta"
df_merged2.to_stata(stata_path, write_index=False)

In [56]:
missing_growth = df_merged[df_merged['revenue_growth'].isna()]

# 查看这些行中 treated == 1 的公司数量
missing_growth[missing_growth['treated'] == 1]

,stock_id,year,IndustryCode,treated,post,total_score_norm,revenue_growth,size,debt_ratio,nwc,roe,roa,firm_id
5610,300389,2014,C39,1,0,0.642857,NaN,20.935853,0.000000,0.382720,0.215367,0.128424,936
5676,300403,2014,C38,1,0,0.500000,NaN,20.938702,0.000000,0.701863,0.114873,0.105781,947
8436,603600,2014,C21,1,0,0.357143,NaN,20.030691,0.089949,0.117063,0.209564,0.108629,1407


In [57]:
df_merged

,stock_id,year,IndustryCode,treated,post,total_score_norm,revenue_growth,size,debt_ratio,nwc,roe,roa,firm_id
0,000012,2014,C30,0,0,0.678571,-0.089127,23.450098,0.153159,-0.275356,0.095987,0.051551,1
1,000012,2015,C30,0,0,0.750000,0.054849,23.474195,0.345932,-0.185206,0.069697,0.034020,1
2,000012,2016,C30,0,0,0.392857,0.207673,23.565078,0.318224,-0.251350,0.102155,0.046523,1
3,000012,2017,C30,0,0,0.571429,0.212313,23.695474,0.269196,-0.129012,0.097580,0.042252,1
4,000012,2018,C30,0,1,0.571429,-0.024766,23.673699,0.274056,-0.066293,0.049759,0.023698,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8545,603998,2015,C27,0,0,0.642857,0.127391,20.787812,0.000000,0.565850,0.102409,0.085008,1425
8546,603998,2016,C27,0,0,0.607143,0.140214,20.963072,0.051298,0.248604,0.075132,0.054882,1425
8547,603998,2017,C27,0,0,0.500000,0.349106,21.040223,0.048283,0.168147,0.060611,0.043469,1425
8548,603998,2018,C27,0,1,0.714286,0.458272,21.219582,0.109578,0.139770,0.069255,0.044436,1425


In [58]:
df_merged[df_merged['treated']==1]

,stock_id,year,IndustryCode,treated,post,total_score_norm,revenue_growth,size,debt_ratio,nwc,roe,roa,firm_id
264,000541,2014,C38,1,0,0.535714,0.214495,22.041470,0.000000,0.469864,0.087409,0.071219,45
265,000541,2015,C38,1,0,0.500000,-0.062563,22.523042,0.000000,0.263375,0.010631,0.008830,45
266,000541,2016,C38,1,0,0.464286,0.170266,22.531582,0.000000,0.409842,0.214878,0.175789,45
267,000541,2017,C38,1,0,0.500000,0.128840,22.459479,0.000000,0.436282,0.154905,0.130432,45
268,000541,2018,C38,1,1,0.464286,0.000465,22.443917,0.000000,0.413620,0.087426,0.067574,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8437,603600,2015,C21,1,0,0.357143,0.183259,20.499499,0.000000,0.382930,0.165101,0.114070,1407
8438,603600,2016,C21,1,0,0.464286,0.234355,20.661760,0.000000,0.293631,0.193097,0.127851,1407
8439,603600,2017,C21,1,0,0.464286,0.312935,20.825287,0.000000,0.252831,0.150263,0.090476,1407
8440,603600,2018,C21,1,1,0.357143,0.309921,21.285571,0.001297,0.354681,0.088555,0.059220,1407
